# 1. Run Temp _View_

## 1.1 Loading Temp View

In [0]:
%run ./_seed_substation_specs

# 2. Creating table

## 2.1 Adding Constraints

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cpt_utility_catalog.gold.dim_substation(
    substation_key BIGINT NOT NULL,
    substation_name STRING NOT NULL,
    capacity_MVA INT NOT NULL,
    voltage_KV INT NOT NULL,
    is_active BOOLEAN NOT NULL,

    CONSTRAINT pk_dim_substation PRIMARY KEY (substation_key) RELY
)

## 2.2 Populating Table

In [0]:
%sql
INSERT OVERWRITE TABLE cpt_utility_catalog.gold.dim_substation

WITH distinct_substations AS (
    SELECT DISTINCT 
        substation 
    FROM cpt_utility_catalog.silver.silver_substations_cleaned
    WHERE substation IS NOT NULL
),

mapped_substations AS (
    SELECT 
        xxhash64(LOWER(TRIM(s.substation))) AS substation_key,
        s.substation AS substation_name,
        COALESCE(ref.capacity, -1.0) AS capacity_MVA, 
        COALESCE(ref.voltage, -1.0)  AS voltage_KV,   
        TRUE AS is_active
    FROM distinct_substations s
    LEFT JOIN tv_substation_specs ref 
           ON LOWER(TRIM(s.substation)) = LOWER(TRIM(ref.substation_name))
)

SELECT * FROM mapped_substations

UNION ALL

-- Mandatory Kimball Fallback Record
SELECT 
    xxhash64('unmapped') AS substation_key,
    'Unmapped'           AS substation_name,
    -1.0                 AS capacity_MVA,
    -1.0                 AS voltage_KV,
    TRUE                 AS is_active;